In [8]:
from langgraph.graph import StateGraph,START, END # type: ignore
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI 
from dotenv import load_dotenv
from typing import TypedDict

load_dotenv()

True

In [9]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.9)

In [10]:
class mainstate(TypedDict):
    query : str
    answer : str
    translated : str 

def translate_to_french(state: mainstate):
    input = state['answer']
    prompt = PromptTemplate(
        input_variables=["input"],
        template="Translate the following English text to French: {input}"
    )
    response = llm.invoke(prompt.format(input=input)).content
    return {"translated" : response}

In [11]:
graph = StateGraph(mainstate)
graph.add_node("translate", translate_to_french)
graph.add_edge(START,"translate")
graph.add_edge("translate", END)
Subgraph = graph.compile()

In [12]:
def answer_query(state: mainstate):
    query = state['query']
    prompt = PromptTemplate(
        input_variables=["query"],
        template="Answer the following question: {query}"
    )
    response = llm.invoke(prompt.format(query=query)).content
    return {"answer" : response}

main_graph = StateGraph(mainstate)
main_graph.add_node("answer_query", answer_query)
main_graph.add_node("translate", Subgraph)

main_graph.add_edge(START,"answer_query")
main_graph.add_edge("answer_query", "translate")
main_graph.add_edge("translate", END)

workflow = main_graph.compile()


In [13]:
workflow.invoke({"query" : "what is full form of SIM?"})

{'query': 'what is full form of SIM?',
 'answer': 'The full form of **SIM** is **Subscriber Identity Module**.',
 'translated': "Le nom complet de **SIM** est **Module d'identité d'abonné**."}